# Xây dựng giao diện giao tiếp cơ bản cho agent

Trong bài học này, bạn sẽ học cách kết nối một LangChain agent ở backend với giao diện chat React ở frontend bằng cách sử dụng **CopilotKit** và giao thức **AG-UI**. Đây là mô hình chuẩn mực giúp bạn có thể kết nối bất kỳ backend agent nào với bất kỳ frontend nào một cách liền mạch.

---

## 📋 Mục tiêu bài học

1. **Khởi chạy LangChain agent:** Khởi động một backend tương thích với AG-UI bằng FastAPI.
2. **Thiết lập CopilotKit:** Kết nối frontend React với agent của bạn thông qua `CopilotRuntime`.
3. **Chuyển đổi linh hoạt giữa các backend:** Chuyển đổi qua lại giữa LangChain/OpenAI và Google ADK/Gemini mà không cần thay đổi bất kỳ dòng code UI nào.

---

## 1. Chuẩn bị môi trường

> 💡 **Lưu ý:** Bạn sẽ cần cài đặt thư viện bằng `pip install -r requirements.txt`, sau đó chạy `npm install` trong thư mục `frontend/`. Đồng thời, tự trang bị `OPENAI_API_KEY` và `GEMINI_API_KEY`.

Đầu tiên, chúng ta cần load các API keys. Trong thực tế, bạn sẽ lấy API key từ OpenAI hoặc Google AI Studio và lưu vào file `.env`.

In [1]:
# Tắt các cảnh báo không cần thiết
import warnings
warnings.filterwarnings("ignore")

# Load các API key (OpenAI và Gemini)
from helper import load_api_keys
load_api_keys()

✓ OpenAI API key loaded
✓ Google API key loaded


---

## 2. Xây dựng backend agent

### 2.1. Khởi động server FastAPI

CopilotKit kết nối với agent của bạn thông qua một HTTP endpoint tương thích với AG-UI. Dưới đây, chúng ta sẽ khởi chạy một server FastAPI và mount `LangGraphAGUIAgent` vào đó.

In [2]:
from fastapi import FastAPI

# Các thư viện của CopilotKit và AG-UI dành cho server
from ag_ui_langgraph import add_langgraph_fastapi_endpoint
from copilotkit import LangGraphAGUIAgent
from langchain.agents import create_agent

# Hàm helper giúp khởi động server và quản lý xung đột port
from helper import start_server

# Tích hợp AG-UI endpoint vào ứng dụng FastAPI
app = FastAPI()
graph = create_agent("openai:gpt-4.1")
agent = LangGraphAGUIAgent(
    name="demo_agent",
    description="Demo agent",
    graph=graph,
)
add_langgraph_fastapi_endpoint(app=app, agent=agent, path="/")

# Khởi động backend server ở port 8002
start_server(app, port=8002)

✓ Server running at http://localhost:8002


### 2.2. Định nghĩa cấu hình cho agent

Bước tiếp theo, bạn tạo một LangChain agent sử dụng model của OpenAI, kết hợp với bộ nhớ lưu trữ (memory checkpointer) và đặc biệt là **CopilotKit middleware**.

In [3]:
from copilotkit import CopilotKitMiddleware
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

# Tạo một LangChain agent
graph = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[],
    middleware=[CopilotKitMiddleware()], # Quan trọng: Kết nối agent với frontend
    checkpointer=MemorySaver(),
    system_prompt=("Bạn là một trợ lý hữu ích."),
)

# Cập nhật graph cho agent (Hỗ trợ hot-reload không cần khởi động lại server)
agent.graph = graph
print("✓ Graph của agent đã được cập nhật!")

✓ Graph của agent đã được cập nhật!


> 🔍 **Tại sao cần `CopilotKitMiddleware()`?**
> Middleware này chính là cầu nối cho phép agent khám phá và gọi các "công cụ frontend" (sẽ được học ở bài sau). Nếu không có middleware này, agent sẽ chỉ nhìn thấy và sử dụng được các công cụ được định nghĩa thuần túy ở backend.

---

## 3. Tích hợp CopilotKit vào frontend

CopilotKit cung cấp cả UI hoàn toàn ẩn (headless) lẫn các component React được xây dựng sẵn. Trong bài này, chúng ta dùng 3 thành phần chính:
1. **`CopilotRuntime`**: Cầu nối bảo mật giữa frontend và backend agent.
2. **`CopilotKit`**: React provider cấu hình kết nối runtime cho toàn bộ app.
3. **`CopilotChat`**: Giao diện chat hoàn chỉnh, có thể tùy biến dễ dàng.

### 3.1. Thiết lập `CopilotRuntime` (Phía Node.js/Server-side frontend)

Tạo file `server.ts` để đăng ký LangChain agent của bạn với tên gọi là `default`.

In [4]:
%%writefile frontend/server.ts
import { serve } from "@hono/node-server";
import { LangGraphHttpAgent } from "@ag-ui/langgraph";
import { CopilotRuntime, createCopilotEndpoint } from "@copilotkit/runtime/v2";

// Trỏ tới backend FastAPI port 8002 vừa tạo
const langGraphAgent = new LangGraphHttpAgent({
  url: process.env.LANGGRAPH_DEPLOYMENT_URL || "http://localhost:8002",
});

// Đăng ký agent vào runtime
const runtime = new CopilotRuntime({
  agents: {
    default: langGraphAgent,
  },
});

// Tạo endpoint cho CopilotKit
const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4002 }, () => {
  console.log("CopilotKit API server running at http://localhost:4002");
});

Overwriting frontend/server.ts


### 3.2. Bọc ứng dụng bằng `CopilotKit` provider

Chỉnh sửa file `main.tsx` trong React để cung cấp URL kết nối cho toàn bộ ứng dụng.

In [5]:
%%writefile frontend/src/main.tsx
import { StrictMode } from "react";
import { createRoot } from "react-dom/client";
import { CopilotKit } from "@copilotkit/react-core/v2";
import "@copilotkit/react-core/v2/styles.css";
import "./globals.css";
import App from "./App";

createRoot(document.getElementById("root")!).render(
  <StrictMode>
    <main className="h-screen w-screen">
      <CopilotKit runtimeUrl="/api/copilotkit" useSingleEndpoint={false}>
        <App />
      </CopilotKit>
    </main>
  </StrictMode>,
);

Overwriting frontend/src/main.tsx


### 3.3. Thêm component `CopilotChat`

Cuối cùng, hiển thị giao diện chat trong `App.tsx` và chỉ định `agentId` là `"default"` (tên chúng ta đã đăng ký trong runtime).

In [6]:
%%writefile frontend/src/App.tsx
import { CopilotChat } from "@copilotkit/react-core/v2";

const agentId = "default";

export default function App() {
  return <CopilotChat agentId={agentId} />;
}

Overwriting frontend/src/App.tsx


### 3.4. Trải nghiệm thành quả

In [7]:
from helper import start_frontend
start_frontend(port=3002)

Starting frontend on port 3002 ...
✓ App running at http://localhost:3002

Read the logs: /home/cuong-ta/Documents/build-interactive-agents-with-generative-ui/1-building-a-basic-agent-ui/frontend/dev-logs.txt


Sau khi khởi chạy ứng dụng frontend (ở port 3002), bạn sẽ thấy một giao diện chat mượt mà xuất hiện trên trình duyệt. Bạn có thể yêu cầu agent thực hiện các tác vụ, ví dụ: "Viết cho tôi một bài thơ!" và nhận lại kết quả văn bản theo thời gian thực được stream trực tiếp từ backend LangChain.

<img src="images/copilotkit-chat.png" alt="Giao diện chat CopilotKit" style="display: block; margin: 0 auto; max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />

---

## 4. Nâng cao - Kết nối với ADK agent

Sức mạnh thực sự của kiến trúc này nằm ở khả năng **chuyển đổi backend mà không ảnh hưởng đến frontend**. Chúng ta sẽ thêm một backend sử dụng Google ADK và Gemini.

### 4.1. Khởi tạo ADK agent backend

Mở một backend server mới (port 8009) chuyên chạy Gemini:

In [8]:
from fastapi import FastAPI
from ag_ui_adk import ADKAgent, add_adk_fastapi_endpoint
from google.adk.agents import LlmAgent
from helper import start_server

# Cấu hình Gemini model
gemini_agent = LlmAgent(
    name="assistant",
    model="gemini-3.5-flash-lite",
    instruction="Bạn là một trợ lý hữu ích.",
)

adk_agent = ADKAgent(
    adk_agent=gemini_agent,
    app_name="demo_app",
    user_id="demo_user",
    session_timeout_seconds=3600,
    use_in_memory_services=True,
)

app_adk = FastAPI()
add_adk_fastapi_endpoint(app_adk, adk_agent, path="/")

start_server(app_adk, port=8009)

✓ Server running at http://localhost:8009


### 4.2. Đăng ký ADK agent vào `CopilotRuntime`

Cập nhật lại file `server.ts` để runtime quản lý đồng thời cả 2 agent:

In [9]:
%%writefile frontend/server.ts
import { serve } from "@hono/node-server";
import { LangGraphHttpAgent } from "@ag-ui/langgraph";
import { HttpAgent } from "@ag-ui/client";
import { CopilotRuntime, createCopilotEndpoint } from "@copilotkit/runtime/v2";

const langGraphAgent = new LangGraphHttpAgent({
  url: process.env.LANGGRAPH_DEPLOYMENT_URL || "http://localhost:8002",
});

const adkAgent = new HttpAgent({
  url: process.env.ADK_AGENT_URL || "http://localhost:8009",
});

const runtime = new CopilotRuntime({
  agents: {
    default: langGraphAgent,
    gemini: adkAgent, // Đăng ký thêm agent mới
  },
});

const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4002 }, () => {
  console.log("CopilotKit API server running at http://localhost:4002");
});

Overwriting frontend/server.ts


### 4.3. Chuyển đổi agent trên UI chỉ với 1 dòng code

Giờ đây, nếu muốn ứng dụng sử dụng ADK agent thay vì Langchain agent, bạn không cần phải viết lại UI hay xử lý logic gọi API phức tạp. Chỉ cần đổi chuỗi `agentId` trong `App.tsx`:

In [10]:
%%writefile frontend/src/App.tsx

import { CopilotChat } from "@copilotkit/react-core/v2";

export const agentId = "gemini"; // Chuyển từ "default" sang "gemini"

export default function App() {
  return <CopilotChat agentId={agentId} />;
}

Overwriting frontend/src/App.tsx


Lúc này, giao diện của bạn đã được tiếp sức mạnh từ ADK agent backend.

---

## 5. Hiểu về kiến trúc giao thức AG-UI

Thành công của việc chuyển đổi mượt mà ở phần 4 đến từ **AG-UI (Agent-User Interaction)**.

<img src="images/protocols.png" alt="Agent Protocol Stack" style="display: block; margin: 0 auto; max-width: 600px; border: 1px solid #ddd; border-radius: 8px;" />

Nhìn vào kiến trúc, chúng ta thấy một hệ sinh thái được tiêu chuẩn hóa:
*   **Tools ↔ Agent (Giao thức MCP):** Các công cụ giao tiếp với agent thông qua giao thức Model Context Protocol.
*   **Agents ↔ Agent (Giao thức A2A):** Các agent giao tiếp đa nhiệm với nhau qua Agent-to-Agent protocol.
*   **Agent ↔ Users (Giao thức AG-UI):** Đây chính là thứ chúng ta vừa áp dụng. AG-UI là giao thức sự kiện (event-based) mở, chạy trên nền HTTP. Nó tiêu chuẩn hóa cách hệ thống truyền tải:
    *   Tin nhắn chat.
    *   Lời gọi hàm/công cụ.
    *   Cập nhật trạng thái.
    *   Streaming dữ liệu theo thời gian thực.

**Lợi ích cốt lõi:**
Nhờ AG-UI, CopilotKit có thể nói chuyện với **bất kỳ backend nào** (LangChain, OpenAI, Google ADK, v.v.) miễn là backend đó tuân thủ tiêu chuẩn AG-UI. Bạn có trải nghiệm thống nhất về streaming và cách hoạt động của công cụ bất kể bạn dùng framework AI nào ở đằng sau.

---

## 🎯 Tổng kết

* Bạn đã biết cách đóng gói một `LangChain` agent thành một HTTP endpoint tương thích với AG-UI.
* Bạn đã nắm được cách gắn `CopilotRuntime`, `CopilotKit` provider, và `CopilotChat` vào một dự án React.
* Bạn đã trực tiếp trải nghiệm sự linh hoạt của kiến trúc tách rời - thay đổi toàn bộ não bộ AI ở backend mà frontend không cần thay đổi bất cứ logic giao diện nào.

**Bước tiếp theo:** Chúng ta sẽ tiến tới **GenUI được kiểm soát**, học cách đăng ký các công cụ frontend (`useComponent()`) và render các kết quả từ agent dưới dạng các component React phong phú thay vì chỉ là văn bản thuần túy.